# Scoring de Satisfaction des Employes — Modele a Risque Minimal (AI Act)

**Risque minimal** — Outil analytique interne RH sans impact individuel direct

## 0. Dependances optionnelles

In [ ]:
# Author: Octo Technology MLOps Tribe
%pip install shap pandera --quiet

## 1. Imports

In [ ]:
# Author: Octo Technology MLOps Tribe
import json
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pandera as pa
from mlflow.models.signature import infer_signature
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
ARTIFACTS_DIR = Path(tempfile.mkdtemp())
print(f"Repertoire artefacts temporaires : {ARTIFACTS_DIR}")

## 2. Configuration MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
PROJECT_NAME = "Employee-Attrition-Prediction"
MLFLOW_TRACKING_URI = f"http://model-platform.com/registry/{PROJECT_NAME}/"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("employee_satisfaction_scoring")
print(f"MLflow tracking URI : {MLFLOW_TRACKING_URI}")

## 3. Generation des donnees synthetiques

In [ ]:
# Author: Octo Technology MLOps Tribe
np.random.seed(42)
N = 5000

tenure_years             = np.random.randint(0, 30, N)
performance_score        = np.random.uniform(3, 10, N).round(2)
salary_k                 = np.random.uniform(25, 120, N).round(1)
num_promotions           = np.random.randint(0, 6, N)
avg_weekly_hours         = np.random.normal(42, 8, N).clip(20, 80).round(1)
num_projects             = np.random.randint(1, 12, N)
distance_from_office_km  = np.random.exponential(scale=20, size=N).clip(0, 150).round(1)
manager_rating           = np.random.uniform(2, 10, N).round(2)

satisfaction = (
    4.0
    + 0.1 * performance_score
    + 0.015 * salary_k
    + 0.2 * num_promotions
    - 0.03 * avg_weekly_hours
    - 0.008 * distance_from_office_km
    + 0.15 * manager_rating
    - 0.02 * num_projects
    + np.random.normal(0, 0.5, N)
).clip(0, 10).round(2)

FEATURES = [
    "tenure_years", "performance_score", "salary_k", "num_promotions",
    "avg_weekly_hours", "num_projects", "distance_from_office_km", "manager_rating",
]
TARGET = "satisfaction_score"
PROTECTED_ATTRIBUTES = []

data_dict = {
    "tenure_years": tenure_years, "performance_score": performance_score,
    "salary_k": salary_k, "num_promotions": num_promotions,
    "avg_weekly_hours": avg_weekly_hours, "num_projects": num_projects,
    "distance_from_office_km": distance_from_office_km,
    "manager_rating": manager_rating, "satisfaction_score": satisfaction,
}

df = pd.DataFrame(data_dict)
print(f"Dataset : {len(df):,} lignes | Cible moyenne : {df[TARGET].mean():.2f} | Std : {df[TARGET].std():.2f}")

## 4. Validation Pandera & statistiques descriptives

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA = pa.DataFrameSchema(
    name="employee_satisfaction_input_schema",
    description="Contrat de donnees — Scoring de Satisfaction des Employes — Modele a Risque Minimal (AI Act)",
    columns={
        "tenure_years":            pa.Column(int,   checks=pa.Check.in_range(0, 40),   nullable=False, description="Anciennete (annees)"),
        "performance_score":       pa.Column(float, checks=pa.Check.in_range(0, 10),   nullable=False, description="Score de performance (0-10)"),
        "salary_k":                pa.Column(float, checks=pa.Check.in_range(20, 200), nullable=False, description="Salaire annuel (kEUR)"),
        "num_promotions":          pa.Column(int,   checks=pa.Check.in_range(0, 10),   nullable=False, description="Nombre de promotions"),
        "avg_weekly_hours":        pa.Column(float, checks=pa.Check.in_range(20, 80),  nullable=False, description="Heures/semaine"),
        "num_projects":            pa.Column(int,   checks=pa.Check.in_range(1, 20),   nullable=False, description="Nombre de projets"),
        "distance_from_office_km": pa.Column(float, checks=pa.Check.in_range(0, 200),  nullable=False, description="Distance bureau (km)"),
        "manager_rating":          pa.Column(float, checks=pa.Check.in_range(0, 10),   nullable=False, description="Note du manager (0-10)"),
        "satisfaction_score":      pa.Column(float, checks=pa.Check.in_range(0, 10),   nullable=False, description="Score de satisfaction cible (0-10)"),
    },
    coerce=False,
    strict=True,
)

try:
    SCHEMA.validate(df, lazy=True)
    PANDERA_STATUS = "PASS"
    PANDERA_ERRORS = 0
    print("Validation Pandera : SUCCES")
except pa.errors.SchemaErrors as exc:
    PANDERA_STATUS = "FAIL"
    PANDERA_ERRORS = len(exc.failure_cases)
    print(f"Validation Pandera : ECHEC ({PANDERA_ERRORS} erreurs)")

In [ ]:
# Author: Octo Technology MLOps Tribe
print("=== Statistiques descriptives ===")
display(df[FEATURES + [TARGET]].describe().round(3))
print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "Aucune valeur manquante.")

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA_YAML_EXPORTED = False
try:
    schema_yaml_path = ARTIFACTS_DIR / "pandera_schema.yaml"
    with open(schema_yaml_path, "w") as f:
        f.write(SCHEMA.to_yaml())
    SCHEMA_YAML_EXPORTED = True
except Exception as e:
    print(f"Export YAML non disponible : {e}")

validation_report = {
    "schema_name": SCHEMA.name,
    "validation_status": PANDERA_STATUS,
    "validation_errors": PANDERA_ERRORS,
    "n_rows_validated": int(len(df)),
    "protected_attributes": PROTECTED_ATTRIBUTES,
    "target_stats": {"mean": round(float(df[TARGET].mean()), 4), "std": round(float(df[TARGET].std()), 4)},
}
validation_report_path = ARTIFACTS_DIR / "data_validation_report.json"
with open(validation_report_path, "w") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print("Rapport de validation exporte.")

## 5. Pretraitement

In [ ]:
# Author: Octo Technology MLOps Tribe
X, y = df[FEATURES], df[TARGET]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
X_val_sc   = pd.DataFrame(scaler.transform(X_val),       columns=FEATURES)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES)

print(f"Train : {len(X_train):,}  |  Validation : {len(X_val):,}  |  Test : {len(X_test):,}")

## 6. Entrainement du modele

In [ ]:
# Author: Octo Technology MLOps Tribe
PARAMS = {
    "alpha":         1.0,
    "fit_intercept": True,
}

model = Ridge(**PARAMS)
model.fit(X_train_sc, y_train)

val_mae = mean_absolute_error(y_val, model.predict(X_val_sc))
print(f"MAE validation : {val_mae:.4f}")
print("Entrainement termine.")

## 7. Evaluation sur le jeu de test

In [ ]:
# Author: Octo Technology MLOps Tribe
y_pred = model.predict(X_test_sc)

METRICS = {
    "mae":  round(float(mean_absolute_error(y_test, y_pred)), 4),
    "mse":  round(float(mean_squared_error(y_test, y_pred)),  4),
    "rmse": round(float(mean_squared_error(y_test, y_pred) ** 0.5), 4),
    "r2":   round(float(r2_score(y_test, y_pred)), 4),
}

print("\n".join(f"{k:<8}: {v}" for k, v in METRICS.items()))

residuals = y_test.values - y_pred
print(f"\nResidus — mean : {residuals.mean():.4f}  |  std : {residuals.std():.4f}")

In [ ]:
# Author: Octo Technology MLOps Tribe
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_pred, y_test.values, alpha=0.4, color="teal", s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=1)
axes[0].set_xlabel("Valeurs predites")
axes[0].set_ylabel("Valeurs reelles")
axes[0].set_title("Predit vs Reel — Satisfaction scorer")

residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=40, color="teal", alpha=0.7, edgecolor="white")
axes[1].axvline(0, color="black", lw=1, linestyle="--")
axes[1].set_xlabel("Residu (reel - predit)")
axes[1].set_ylabel("Frequence")
axes[1].set_title("Distribution des residus")

plt.tight_layout()
roc_path = ARTIFACTS_DIR / "residuals_plot.png"
plt.savefig(roc_path, dpi=150)
plt.show()

## 8. Analyse d'equite (Fairness)

In [ ]:
# Author: Octo Technology MLOps Tribe
# Pour un modele de regression, l'equite se mesure par les erreurs par sous-groupe
df_eval = X_test.copy()
df_eval["target"]    = y_test.values
df_eval["predicted"] = y_pred
df_eval["residual"]  = df_eval["target"] - df_eval["predicted"]

# Analyse par attribut protege si disponible
FAIRNESS_REPORT = {
    "protected_attributes": PROTECTED_ATTRIBUTES,
    "note": "Analyse d'equite par sous-groupe sur les metriques de regression (MAE, RMSE).",
    "global_mae":  round(float(mean_absolute_error(df_eval["target"], df_eval["predicted"])), 4),
    "global_rmse": round(float(mean_squared_error(df_eval["target"], df_eval["predicted"]) ** 0.5), 4),
    "subgroups": {},
}

for attr in PROTECTED_ATTRIBUTES:
    if attr in df_eval.columns:
        subgroup_results = {}
        for val, group in df_eval.groupby(attr, observed=True):
            if len(group) >= 20:
                subgroup_results[str(val)] = {
                    "n":    int(len(group)),
                    "mae":  round(float(mean_absolute_error(group["target"], group["predicted"])), 4),
                    "rmse": round(float(mean_squared_error(group["target"], group["predicted"]) ** 0.5), 4),
                }
        FAIRNESS_REPORT["subgroups"][attr] = subgroup_results
        print(f"=== Equite par {attr} ===")
        print(pd.DataFrame(subgroup_results).T.to_string())

fairness_path = ARTIFACTS_DIR / "fairness_report.json"
with open(fairness_path, "w") as f:
    json.dump(FAIRNESS_REPORT, f, indent=2, ensure_ascii=False)
print("Rapport d'equite exporte.")

## 9. Explicabilite

In [ ]:
# Author: Octo Technology MLOps Tribe
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    imp_label = "Importance des variables (Gini)"
elif hasattr(model, "coef_"):
    coef = model.coef_
    importances = np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef[0])
    imp_label = "Importance des variables (|coef| moyen)"
else:
    importances = None

if importances is not None:
    fi_df = pd.DataFrame({"feature": FEATURES, "importance": importances}).sort_values("importance", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#c0392b" if f in PROTECTED_ATTRIBUTES else "teal" for f in fi_df["feature"][::-1]]
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=colors)
    ax.set_title(f"{imp_label} — rouge = attribut protege")
    ax.set_xlabel("Importance relative")
    plt.tight_layout()
    fi_path = ARTIFACTS_DIR / "feature_importance.png"
    plt.savefig(fi_path, dpi=150)
    plt.show()
    print(fi_df.to_string(index=False))
else:
    fi_df = None
    print("Aucune methode d'importance disponible pour ce modele.")

In [ ]:
# Author: Octo Technology MLOps Tribe
SHAP_AVAILABLE = False
try:
    import shap
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_sc.iloc[:300])
    plt.figure()
    shap.summary_plot(shap_values, X_test_sc.iloc[:300], show=False, plot_size=(10, 6))
    shap_path = ARTIFACTS_DIR / "shap_summary.png"
    plt.savefig(shap_path, dpi=150, bbox_inches="tight")
    plt.close()
    SHAP_AVAILABLE = True
    print("SHAP summary plot genere.")
except ImportError:
    print("SHAP non disponible.")

## 10. Preparation des artefacts de documentation

In [ ]:
# Author: Octo Technology MLOps Tribe
fi_csv_path = ARTIFACTS_DIR / "feature_importance.csv"
fi_df.to_csv(fi_csv_path, index=False)

pp_context = {"schema_name": SCHEMA.name, "pandera_status": PANDERA_STATUS, "pandera_errors": PANDERA_ERRORS}
preprocessing_md = Path("preprocessing_description_satisfaction_template.md").read_text(encoding="utf-8").format_map(pp_context)
pp_path = ARTIFACTS_DIR / "preprocessing_description.md"
pp_path.write_text(preprocessing_md, encoding="utf-8")

print("Artefacts prepares :")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  {p.name}")

## 11. Logging MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
MODEL_NAME  = "satisfaction_scorer"
TEAM        = "mlops-tribe"
ENVIRONMENT = "staging"

model_card_text = Path("model_card_satisfaction.md").read_text(encoding="utf-8")
model_card_path = ARTIFACTS_DIR / "model_card.md"
model_card_path.write_text(model_card_text, encoding="utf-8")

with mlflow.start_run(run_name=f"{MODEL_NAME}_v1") as run:
    mlflow.log_params(PARAMS)
    mlflow.log_param("scaler",           "StandardScaler")
    mlflow.log_param("feature_count",    len(FEATURES))
    mlflow.log_param("features",         ", ".join(FEATURES))
    mlflow.log_param("train_size",       len(X_train))
    mlflow.log_param("val_size",         len(X_val))
    mlflow.log_param("test_size",        len(X_test))

    mlflow.log_metrics(METRICS)
    mlflow.log_metric("mae_validation",      round(val_mae, 4))
    mlflow.log_metric("dataset_total_size",  N)

    mlflow.set_tag("model_type",             "Ridge")
    mlflow.set_tag("framework",              "scikit-learn")
    mlflow.set_tag("data_source",            "Donnees synthetiques RH")
    mlflow.set_tag("contains_personal_data", "non — donnees agreges synthetiques")
    mlflow.set_tag("protected_attributes",   "aucun")
    mlflow.set_tag("threshold_mae",          "0.8")
    mlflow.set_tag("threshold_r2",           "0.75")

    mlflow.set_tag("model.author",      "Octo Technology MLOps Tribe")
    mlflow.set_tag("model.team",        TEAM)
    mlflow.set_tag("model.environment", ENVIRONMENT)
    mlflow.set_tag("data.synthetic",    "true")
    mlflow.set_tag("pandera.status",    PANDERA_STATUS)
    mlflow.set_tag("ai_act.risk_level", "minimal")
    mlflow.set_tag("ai_act.annex_ref",  "N/A")
    mlflow.set_tag("ai_act.domain",     "Analytique RH interne")
    mlflow.set_tag("mlflow.note.content", model_card_text)

    for art in [fi_csv_path, fi_path, fairness_path, roc_path, pp_path, validation_report_path, model_card_path]:
        mlflow.log_artifact(str(art))
    if SCHEMA_YAML_EXPORTED:
        mlflow.log_artifact(str(schema_yaml_path))
    if SHAP_AVAILABLE:
        mlflow.log_artifact(str(shap_path))

    signature     = infer_signature(X_train_sc, model.predict(X_train_sc))
    input_example = X_train_sc.head(3)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="custom_model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=input_example,
    )
    run_id = run.info.run_id

print(f"\nRun MLflow : {run_id}")
print(f"   Modele enregistre : {MODEL_NAME}")
print(f"   MAE : {METRICS['mae']}  |  R2 : {METRICS['r2']}  |  RMSE : {METRICS['rmse']}")